# Risk-Aware Inventory Redistribution — pipeline walkthrough (local or Google Colab)

This notebook **does not duplicate the implementation**: it calls the reusable modules in `src/inventory_rl`
and the stage scripts in `scripts/`. All results are produced by the same code as the command-line pipeline.

**Colab setup:** upload the `prototype` folder (without `.venv`) to Google Drive, and upload the three original
dataset ZIP archives to a Drive folder. Then set the two paths in the next cell. The original archives are only
read, never modified.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/prototype")              # <- adjust
    os.environ["INVENTORY_DATA_DIR"] = "/content/drive/MyDrive/dataset"  # <- folder with the ZIP archives
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")])
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
print(PROJECT)

## 1. Extract (safe, non-destructive) and audit the raw data

In [ ]:
from inventory_rl.data.extract import extract_all
for r in extract_all():
    print(r.source, r.target_dir, r.extracted or r.skipped_existing, r.rejected)

In [ ]:
subprocess.run([sys.executable, "scripts/run_data_audit.py"], check=True)
import json
audit = json.load(open("results/reports/data_audit.json"))
{k: audit["mendeley"]["inventory"][k] for k in ["rows", "open_records", "negative_qty_rows", "share_stock_change_equals_minus_sales"]}

## 2. Weekly store × SKU panel and leakage-safe features

In [ ]:
subprocess.run([sys.executable, "scripts/run_preprocess.py"], check=True)
import pandas as pd
panel = pd.read_parquet("data/processed/panel.parquet")
panel.head()

## 3–4. Rolling-origin forecasts and prediction intervals

In [ ]:
subprocess.run([sys.executable, "scripts/run_forecasting.py"], check=True)
pd.DataFrame(json.load(open("results/metrics/forecast_metrics.json"))["test"]).T.round(4)

In [ ]:
from IPython.display import Image, display
display(Image("results/figures/forecast_vs_actual_total.png"))

## 5–6. Inventory risk, store network (simulated locations)

In [ ]:
subprocess.run([sys.executable, "scripts/run_risk.py"], check=True)
risk = pd.read_parquet("data/processed/risk.parquet")
risk[risk.split == "test"].risk_class.value_counts()

## 7–8. Simulator and PPO training
Training all seeds takes ~25 min on a laptop CPU. For a quick demo pass `--timesteps 20000 --seeds 0`.

In [ ]:
subprocess.run([sys.executable, "scripts/run_rl.py", "--timesteps", "20000", "--seeds", "0"], check=True)

## 9–10. Evaluation on the held-out test weeks

In [ ]:
subprocess.run([sys.executable, "scripts/run_evaluation.py"], check=True)
s = pd.read_csv("results/metrics/evaluation_summary.csv")
s[["policy", "scenario", "fill_rate", "lost_units", "transfer_cost", "saving_vs_noop", "saving_vs_noop_std"]].round(2)

## Using the modules directly
Example: one simulated week for one SKU with the rule-based (upper-bound) policy.

In [ ]:
from inventory_rl.utils.config import load_config
from inventory_rl.environment.factory import load_context
from inventory_rl.baselines.rules import RuleBasedPolicy
ctx = load_context(load_config())
sim = ctx.sim("test", "replay")
prod = sim.products[0]
sim.reset(prod, 0)
transfers = RuleBasedPolicy("upper")(sim)
print(prod, transfers)
print(sim.step(transfers))